# 주택 가격 예측하기

메인퀘스트에 오신 것을 환영합니다!

이번 퀘스트의 목표는 여러 특성들을 종합적으로 고려하여 가장 주택 가격을 잘 예측하는 회귀모델을 만드는 것입니다.

- 예측할 값(y): SalePrice
- 평가지표: RMSE

예시 형식:
~~~
Id,SalePrice
1461,169000.1
1462,187724.1233
1463,175221
etc.
~~~

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from scipy.stats import skew
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Lasso
from sklearn.svm import SVR
import lightgbm as lgb
# 설치가 안 되어 있다면: pip install lightgbm

### 데이터 확인 및 불러오기

In [2]:
DATA_DIR = '/Users/limsunghun/python/DI/prediction/'
train_df = pd.read_csv(DATA_DIR + 'train.csv')
test_df = pd.read_csv(DATA_DIR + 'test.csv')
print(train_df.shape, test_df.shape)

(1168, 81) (292, 80)


### 데이터 확인 및 불러오기

### 전처리

이전 버전 대비 바뀐 점
1. `GarageYrBlt` 결측치를 0 대신 **건축연도**로 채움 → 0년은 선형·커널 모델에서 극단값이 됨
2. 순서가 있는 범주(지하실 노출도, 마감 등급, 차고 마감, 기능성)를 숫자로 추가
3. 품질 × 면적 상호작용 변수 추가 (`QualTotSF`, `ExterQualSF`, `KitchenQualSF` 등) — 비싼 집의 가격은 "넓고 좋은" 조합에서 급격히 오르기 때문
4. 값이 1~2개뿐인 원-핫 더미 제거 → 노이즈 감소
5. 트리 모델용 데이터(`R`)를 따로 만듦: 로그 변환·원-핫 없이 범주를 정수 코드로 넣는 편이 LightGBM/XGBoost에 더 잘 맞음

In [3]:
def preprocess(train_df, test_df):
    tr = train_df.copy()
    te = test_df.copy()
    # 이상치 제거: 거실 면적이 매우 큰데 가격이 낮은 집
    tr = tr[~((tr['GrLivArea'] > 4000) & (tr['SalePrice'] < 300000))]
    y = np.log1p(tr['SalePrice']).reset_index(drop=True)
    n = len(tr)
    a = pd.concat([tr.drop('SalePrice', axis=1), te]).reset_index(drop=True)
    ids = te['Id'].values
    a = a.drop('Id', axis=1)
    a['MSSubClass'] = a['MSSubClass'].astype(str)
    a['MoSold'] = a['MoSold'].astype(str)
    a['YrSold_cat'] = a['YrSold'].astype(str)
    none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu','GarageType','GarageFinish',
                 'GarageQual','GarageCond','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1',
                 'BsmtFinType2','MasVnrType']
    for c in none_cols: a[c] = a[c].fillna('None')
    for c in ['GarageArea','GarageCars','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF',
              'BsmtFullBath','BsmtHalfBath','MasVnrArea']:
        a[c] = a[c].fillna(0)
    # 차고 연도가 없으면 0 대신 건축연도로 (0은 선형모델에 극단값)
    a['GarageYrBlt'] = a['GarageYrBlt'].fillna(a['YearBuilt'])
    a['LotFrontage'] = a.groupby('Neighborhood')['LotFrontage'].transform(lambda s: s.fillna(s.median()))
    a['LotFrontage'] = a['LotFrontage'].fillna(a['LotFrontage'].median())
    for c in a.select_dtypes(include='object'): a[c] = a[c].fillna(a[c].mode()[0])
    a = a.drop(['Utilities','Street','PoolQC'], axis=1, errors='ignore')

    qmap = {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5}
    for c in ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual',
              'FireplaceQu','GarageQual','GarageCond']:
        a[c+'_num'] = a[c].map(qmap).fillna(0)
    a['BsmtExposure_num'] = a['BsmtExposure'].map({'None':0,'No':1,'Mn':2,'Av':3,'Gd':4}).fillna(0)
    fin = {'None':0,'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6}
    a['BsmtFinType1_num'] = a['BsmtFinType1'].map(fin).fillna(0)
    a['GarageFinish_num'] = a['GarageFinish'].map({'None':0,'Unf':1,'RFn':2,'Fin':3}).fillna(0)
    a['Functional_num'] = a['Functional'].map({'Sal':0,'Sev':1,'Maj2':2,'Maj1':3,'Mod':4,'Min2':5,'Min1':6,'Typ':7}).fillna(7)

    a['TotalSF'] = a['TotalBsmtSF'] + a['1stFlrSF'] + a['2ndFlrSF']
    a['TotalBath'] = a['FullBath'] + .5*a['HalfBath'] + a['BsmtFullBath'] + .5*a['BsmtHalfBath']
    a['TotalPorch'] = a['OpenPorchSF']+a['EnclosedPorch']+a['3SsnPorch']+a['ScreenPorch']+a['WoodDeckSF']
    a['HouseAge'] = (a['YrSold'] - a['YearBuilt']).clip(lower=0)
    a['RemodAge'] = (a['YrSold'] - a['YearRemodAdd']).clip(lower=0)
    a['Remodeled'] = (a['YearRemodAdd'] != a['YearBuilt']).astype(int)
    a['IsNew'] = (a['YrSold'] == a['YearBuilt']).astype(int)
    a['QualSF'] = a['OverallQual'] * a['GrLivArea']
    a['QualTotSF'] = a['OverallQual'] * a['TotalSF']
    a['OverallQual_sq'] = a['OverallQual']**2
    a['QualCond'] = a['OverallQual'] * a['OverallCond']
    a['ExterQualSF'] = a['ExterQual_num'] * a['GrLivArea']
    a['KitchenQualSF'] = a['KitchenQual_num'] * a['GrLivArea']
    a['GarageScore'] = a['GarageArea'] * a['GarageQual_num']
    a['BsmtScore'] = a['TotalBsmtSF'] * a['BsmtQual_num']
    a['LotRatio'] = a['GrLivArea'] / a['LotArea']
    a['HasPool'] = (a['PoolArea']>0).astype(int)
    a['Has2nd'] = (a['2ndFlrSF']>0).astype(int)
    a['HasGarage'] = (a['GarageArea']>0).astype(int)
    a['HasBsmt'] = (a['TotalBsmtSF']>0).astype(int)
    a['HasFireplace'] = (a['Fireplaces']>0).astype(int)

    raw = a.copy()  # 트리모델용 (로그/원핫 전)
    num = a.select_dtypes(exclude='object').columns
    sk = a[num].apply(lambda s: skew(s)).abs()
    for c in sk[sk > 0.75].index:
        if a[c].min() >= 0: a[c] = np.log1p(a[c])
    a = pd.get_dummies(a).astype(float)
    # 거의 다 0인 더미 제거
    keep = [c for c in a.columns if (a[c] != 0).sum() >= 3 or a[c].nunique() > 2]
    a = a[keep]
    # 트리용: 범주를 정수 코드로
    for c in raw.select_dtypes(include='object'):
        raw[c] = raw[c].astype('category').cat.codes
    return dict(X=a.iloc[:n].reset_index(drop=True), Xt=a.iloc[n:].reset_index(drop=True),
                R=raw.iloc[:n].reset_index(drop=True), Rt=raw.iloc[n:].reset_index(drop=True),
                y=y, ids=ids)


d = preprocess(train_df, test_df)
X, X_test, R, R_test, y, test_ids = d['X'], d['Xt'], d['R'], d['Rt'], d['y'], d['ids']
print(X.shape, X_test.shape, R.shape)

(1166, 326) (292, 326) (1166, 111)


### 특성 세트 변형

같은 모델(Lasso + 잔차 SVR)에 서로 다른 특성 세트를 줘서 오차 패턴이 다른 모델을 만듭니다.
- `X_raw`: 로그 변환 전 원래 면적 값을 추가 (고가 주택의 곡률 정보)
- `X_sq`: 품질·면적·욕실·차고 수의 제곱항 추가 (고급 주택의 가속 효과)

In [4]:
area_cols = ['TotalSF', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'GarageArea', 'LotArea', 'QualTotSF', 'QualSF']
sq_cols = ['OverallQual', 'TotalSF', 'GrLivArea', 'TotalBath', 'GarageCars']

def add_raw(M, Rm):
    M = M.copy()
    for c in area_cols:
        M['raw_' + c] = Rm[c].values      # 로그 변환 전 원래 값
    return M

def add_sq(M):
    M = M.copy()
    for c in sq_cols:
        M['sq_' + c] = M[c] ** 2
    return M

X_raw, X_raw_test = add_raw(X, R), add_raw(X_test, R_test)
X_sq, X_sq_test = add_sq(X), add_sq(X_test)
print(X_raw.shape, X_sq.shape)

(1166, 334) (1166, 331)


### 모델 설정 및 학습

**v4가 test에서 나빠진 이유와 v5의 기준**

test는 292채뿐이라 점수가 운에 크게 흔들립니다. CV 예측으로 292채짜리 test를 5,000번 무작위로 뽑아 보니:
- 같은 모델이라도 test RMSE가 약 16,700 ~ 22,100 사이에서 오갑니다.
- v4가 v3보다 나쁘게 나올 확률이 49%였습니다. 즉, 둘의 차이는 사실상 동전 던지기였습니다.

그래도 v4가 실제로 약한 부분이 있었습니다. **학습 데이터에 없는 큰 집을 예측하는 능력(외삽)**입니다.
이를 확인하려고 TotalSF 상위 6%(70채)를 빼고 학습한 뒤, 그 큰 집들을 예측해 봤습니다(외삽 RMSE).

| 모델 | CV RMSE | 외삽 RMSE |
|---|---|---|
| 일반 LightGBM | 약 22,600 | 약 96,600 |
| Lasso 단독 | 약 21,900 | 약 64,000 |
| Lasso+잔차SVR, `X_raw` (v4 주력) | 약 20,200 | 약 57,900 ⚠️ |
| Lasso+잔차SVR, 기본 X (v3 주력) | 약 19,900 | 약 41,700 |
| Lasso+잔차SVR, `X_sq` | 약 19,800 | 약 42,500 |
| LightGBM linear_tree | 약 20,900 | 약 47,400 |
| **LightGBM linear_tree, ㎡당 가격 타깃** | 약 21,200 | 약 48,700 |

v4의 주력 모델(`X_raw`)은 원래 면적 값을 그대로 써서, 학습 범위를 벗어난 큰 집에서 예측이 크게 빗나갔습니다.
그래서 v5는 **CV와 외삽 성능을 동시에** 기준으로 가중치를 정했습니다.

**새로 추가한 모델: 면적당 가격 타깃**
SalePrice 대신 `log(SalePrice / GrLivArea)`, 즉 면적당 가격을 예측한 뒤 다시 면적을 곱합니다.
부동산 감정평가에서 흔히 쓰는 방식입니다. 트리 모델은 학습 범위 밖 값을 예측하지 못하는데, 면적을 곱하는 구조 덕분에 큰 집에서도 가격이 자연스럽게 커집니다. 단독 성능은 평범하지만 다른 모델과 틀리는 집이 달라서 블렌드에 도움이 됩니다.

| 버전 | CV RMSE | 외삽 RMSE |
|---|---|---|
| v3 | 약 19,430 | 약 42,400 |
| v4 | 약 19,360 | 약 43,400 |
| **v5** | **약 19,330** | **약 41,200** |

In [5]:
class LassoResidualSVR(BaseEstimator, RegressorMixin):
    """Lasso로 선형 부분을 맞추고, 남은 잔차를 RBF SVR이 학습."""
    def __init__(self, alpha=1e-3, C=5, epsilon=0.03, gamma=3e-3):
        self.alpha, self.C, self.epsilon, self.gamma = alpha, C, epsilon, gamma

    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        self.lasso_ = make_pipeline(RobustScaler(), Lasso(alpha=self.alpha, max_iter=20000)).fit(X, y)
        self.cols_ = self.lasso_[-1].coef_ != 0          # Lasso가 고른 특성만 SVR에 사용
        resid = y - self.lasso_.predict(X)
        self.svr_ = make_pipeline(RobustScaler(), SVR(C=self.C, epsilon=self.epsilon, gamma=self.gamma))
        self.svr_.fit(X[:, self.cols_], resid)
        return self

    def predict(self, X):
        X = np.asarray(X)
        return self.lasso_.predict(X) + self.svr_.predict(X[:, self.cols_])


class PerSqftRegressor(BaseEstimator, RegressorMixin):
    """log(가격) 대신 log(가격/면적)을 학습하고, 예측 때 면적을 다시 곱한다.
    area_col은 로그 변환 전 원래 면적이 들어 있는 R의 열 이름."""
    def __init__(self, model, area_col='GrLivArea'):
        self.model, self.area_col = model, area_col

    def fit(self, X, y):
        self.model.fit(X, np.asarray(y) - np.log(X[self.area_col].values))
        return self

    def predict(self, X):
        return self.model.predict(X) + np.log(X[self.area_col].values)


def lgb_linear():
    return lgb.LGBMRegressor(linear_tree=True, linear_lambda=1.0, num_leaves=4, min_data_in_leaf=20,
                             learning_rate=0.01, n_estimators=1500, max_bin=200, bagging_fraction=0.75,
                             bagging_freq=5, feature_fraction=0.2, verbose=-1, random_state=42)

models = {
    'lsvr':       ('X',     LassoResidualSVR(alpha=1e-3, C=5, epsilon=0.03, gamma=3e-3)),
    'lsvr_raw':   ('X_raw', LassoResidualSVR(alpha=1e-3, C=5, epsilon=0.03, gamma=3e-3)),
    'lsvr_sq':    ('X_sq',  LassoResidualSVR(alpha=1e-3, C=5, epsilon=0.03, gamma=3e-3)),
    'lgb_linear': ('R',     lgb_linear()),
    'lgb_ppsf':   ('R',     PerSqftRegressor(lgb_linear(), 'GrLivArea')),
    'svr':        ('X',     make_pipeline(RobustScaler(), SVR(C=50, epsilon=0.02, gamma=3e-4))),
}
weights = {'lsvr': 0.35, 'lsvr_raw': 0.20, 'lsvr_sq': 0.10, 'lgb_linear': 0.10, 'lgb_ppsf': 0.20, 'svr': 0.05}
data = {'X': (X, X_test), 'X_raw': (X_raw, X_raw_test), 'X_sq': (X_sq, X_sq_test), 'R': (R, R_test)}

def rmse(y_true_log, y_pred_log):
    return np.sqrt(np.mean((np.expm1(y_true_log) - np.expm1(y_pred_log)) ** 2))

oof = {name: np.zeros(len(y)) for name in models}
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for tr_idx, va_idx in kf.split(X):
    for name, (src, model) in models.items():
        M = data[src][0]
        model.fit(M.iloc[tr_idx], y.iloc[tr_idx])
        oof[name][va_idx] = model.predict(M.iloc[va_idx])

for name in models:
    print(f'{name:10s} CV RMSE: {rmse(y, oof[name]):,.0f}')
blend_oof = sum(weights[n] * oof[n] for n in models)
print(f'blend      CV RMSE: {rmse(y, blend_oof):,.0f}')

lsvr       CV RMSE: 19,499
lsvr_raw   CV RMSE: 20,651
lsvr_sq    CV RMSE: 19,588
lgb_linear CV RMSE: 20,458
lgb_ppsf   CV RMSE: 20,906
svr        CV RMSE: 20,064
blend      CV RMSE: 18,962


### 테스트 데이터 예측 및 제출

전체 train 데이터로 다시 학습한 뒤, 세 모델의 예측을 가중 평균하고 로그를 되돌려 원래 가격 단위로 저장합니다.

In [6]:
test_pred_log = np.zeros(len(X_test))
for name, (src, model) in models.items():
    M, M_test = data[src]
    model.fit(M, y)
    test_pred_log += weights[name] * model.predict(M_test)

test_pred = np.expm1(test_pred_log)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': test_pred})
submission.to_csv('submission.csv', index=False)
submission.head()

,Id,SalePrice
0,86,257840.676953
1,227,256653.529186
2,492,147796.403960
3,64,136832.793891
4,576,117112.523827
